# 9.4 何時需要安全回應？


兩個人都問「盒子的密碼怎麼處理」：一位在管理自己的公開測試盒，另一位想取用別人未授權的祕密盒。如果只看到「密碼」兩字就拒絕，前者的正常工作也會失敗；如果全部回答，又會跨過我們明定的邊界。本節先用窄小、可核對的規則，教資料標註如何看情境。

前置是[9.1各自保存安全標籤](https://birdhackor.github.io/tiny-perceptron-vlm/9.1.html)、[W.2字典與迴圈](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2)及[8.6的if/else分支](https://birdhackor.github.io/tiny-perceptron-vlm/8.6.html)。這裡的課堂規則是「不提供他人未授權的祕密碼，可以協助自己的公開測試碼」。授權資訊是小世界中可信的欄位，不靠問句裡的一句自我宣稱猜測。這個規則只示範依欄位建立邊界，不是涵蓋所有真實安全問題的分類器。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
examples = [
    {"owner": "他人", "permission": False},
    {"owner": "自己", "permission": True},
]
for row in examples:
    if row["permission"]:
        target = "可以協助整理你的公開測試碼。"
    else:
        target = "無法提供他人的祕密碼；可協助聯絡盒主。"
    print(row, "→", target)

輸入兩筆都與盒子碼有關，差異是`permission`是否為真。它在此代表管理者提供的有效授權，不表示模型能自動驗證外部身分。關鍵`if`讀這個欄位而不是搜尋「密碼」字串。輸出第一筆走拒絕與替代方案，第二筆走允許協助；`owner`同時記錄背景，方便人回查標註是否合理。

這些目標句可以作為後續訓練範例，但規則程式本身只會處理我們提供的欄位。自然語言模型是否在新措辭上讀懂授權差異，要另做測試，不能用這兩行的固定輸出當成成效。真實應用也不能讓外來文本自行把permission改為True，應由應用中的可靠授權流程提供資訊。

成對題目的好處是主題詞保持接近，卻有不同應答方式。讓模型同時見到這些例子，才能避免「盒子」「祕密」「碼」等詞與拒絕機械綁定；這也為[9.6過度拒絕的檢查](https://birdhackor.github.io/tiny-perceptron-vlm/9.6.html)準備了明確對照。

這段資料裡owner與permission彼此相符，所以沒有處理兩欄互相矛盾的情況。若有人標他人卻又設True，可能代表有效委託，也可能是資料錯誤；作者需先寫清規則再標注。不要讓模型從不一致的欄位自行猜我們意思。單一可信permission欄位在小世界裡足夠，而真實授權可能需要多項證明。把教學邊界說窄，反而有助於測出它到底學會了哪一個判斷。

練習只把第一筆`permission`改成`True`，先預測會走允許分支，再重跑。隨後說明這次修改的意義：你改變了小世界的可信授權條件，並沒有證明現實中的陌生使用者一句「我有權限」就足夠。如果任務要區分公開碼與私人碼，先增加並解釋相應可信欄位，才修改規則。
